# Home Credit Default Risk — final model

This notebook runs **only the winning path** found in `main.ipynb` (the full work log with every experiment, including the ones that did not help).

| Step | What it adds | CV AUC | Public LB |
|---|---|---|---|
| Tutorial | Random Forest, 5 features | 0.669 | |
| v4 | LightGBM + cleaning + ratios + implied interest rate + payment signals | 0.7652 | 0.76912 |
| v5 | + CatBoost, 50/50 blend | 0.7665 | 0.76957 |
| v6 | + 3 LightGBM seeds | 0.7669 | 0.7697 |
| v7 | + repeat-client features + model-imputed EXT_SOURCE | 0.7706 | 0.77337 |
| **v8 (this notebook)** | + 3 CatBoost seeds | see end | |

Validation: `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` for every model, so the out-of-fold (OOF) predictions of all models are comparable and can be blended.

Runtime on a 2-core machine: about 1 h 15 min, mostly CatBoost.

In [1]:
import time
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score

SEED = 42
SEEDS = [42, 7, 2026]
np.random.seed(SEED)

train = pd.read_csv("input/train.csv")
test = pd.read_csv("input/test.csv")
sample_sub = pd.read_csv("input/sample_submission.csv")
print("train:", train.shape, "| test:", test.shape)
T0 = time.time()

train: (171202, 34) | test: (61500, 33)


## 1. Cleaning

* `DAYS_EMPLOYED = 365243` (pensioners / unemployed) and `OWN_CAR_AGE >= 64` are codes, not values: set to NaN and keep a flag.
* `XNA` and `Unknown` are disguised missing values.
* `FLAG_MOBIL` is constant, so it is dropped.

In [2]:
def preprocess(df, drop_cols=("FLAG_MOBIL",)):
    """Phase 2 cleaning. Applied identically to train and test."""
    df = df.copy()

    # Codes that are not real values -> NaN, keeping the information in a flag
    anom = df["DAYS_EMPLOYED"] == 365243
    df["DAYS_EMPLOYED_ANOM"] = anom.astype(int)
    df.loc[anom, "DAYS_EMPLOYED"] = np.nan

    car_anom = df["OWN_CAR_AGE"] >= 64
    df["OWN_CAR_AGE_ANOM"] = car_anom.astype(int)
    df.loc[car_anom, "OWN_CAR_AGE"] = np.nan

    # Disguised missing values
    df = df.replace({"XNA": np.nan, "Unknown": np.nan})

    # Useless columns
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    return df

## 2. Loan features

* **Ratios**: how heavy the loan is compared with the client's income and with the price of the goods.
* **Implied interest rate**: Home Credit does not give the interest rate or the number of instalments, but `AMT_CREDIT` and `AMT_ANNUITY` follow the annuity formula. For each common duration we solve for the monthly rate, and keep the shortest duration with a realistic rate (0.1% to 5% per month).
* **Payment signals**: decimals in `AMT_CREDIT / AMT_ANNUITY` come from the interest; round prices and incomes.

In [3]:
def add_ratios(df):       # group B
    df["CREDIT_INCOME_RATIO"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]     # how many yearly salaries they borrow
    df["ANNUITY_INCOME_RATIO"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]   # share of income going to the instalment
    df["CREDIT_TERM"] = df["AMT_ANNUITY"] / df["AMT_CREDIT"]                  # instalment / loan ~ 1 / number of payments
    df["GOODS_CREDIT_RATIO"] = df["AMT_GOODS_PRICE"] / df["AMT_CREDIT"]       # how much of the loan buys the goods
    df["CREDIT_GOODS_DIFF"] = df["AMT_CREDIT"] - df["AMT_GOODS_PRICE"]        # money borrowed beyond the goods price
    df["INCOME_PER_PERSON"] = df["AMT_INCOME_TOTAL"] / df["CNT_FAM_MEMBERS"]


TERMS = [6, 10, 12, 18, 24, 30, 36, 42, 48, 54, 60]

def implied_rate(credit, annuity, n):
    """Monthly rate r such that annuity = credit * r / (1 - (1+r)^-n). Solved by bisection; NaN if impossible."""
    lo = np.full(len(credit), 1e-7)
    hi = np.full(len(credit), 0.2)
    f = lambda r: credit * r / (1 - (1 + r) ** (-n)) - annuity
    possible = annuity * n > credit                 # with no interest you would pay exactly credit
    for _ in range(60):
        mid = (lo + hi) / 2
        v = f(mid)
        hi = np.where(v > 0, mid, hi)
        lo = np.where(v > 0, lo, mid)
    r = (lo + hi) / 2
    r[~possible] = np.nan
    return r

def add_rates(df):
    c, a = df["AMT_CREDIT"].values, df["AMT_ANNUITY"].values
    rates = np.column_stack([implied_rate(c, a, n) for n in TERMS])
    rates[(rates > 0.05) | (rates < 0.001)] = np.nan            # keep realistic monthly rates only
    first = np.argmax(~np.isnan(rates), axis=1)                 # shortest duration with a realistic rate
    has = (~np.isnan(rates)).any(axis=1)
    df["TERM_GUESS"] = np.where(has, np.array(TERMS)[first], np.nan)
    df["RATE_GUESS"] = np.where(has, rates[np.arange(len(rates)), first], np.nan)
    for n in [12, 18, 24, 30, 36]:                              # the rate under each common duration
        r = implied_rate(c, a, n)
        r[(r > 0.05) | (r < 0.001)] = np.nan
        df[f"RATE_{n}M"] = r

def add_payment_signals(df):
    df["N_PAYMENTS"] = df["AMT_CREDIT"] / df["AMT_ANNUITY"]
    df["N_PAYMENTS_FRAC"] = df["N_PAYMENTS"] - np.floor(df["N_PAYMENTS"])   # the decimals come from interest
    df["GOODS_ROUND_1000"] = (df["AMT_GOODS_PRICE"] % 1000 == 0).astype(float)
    df["INCOME_ROUND_10000"] = (df["AMT_INCOME_TOTAL"] % 10000 == 0).astype(int)

## 3. Repeat clients

All `DAYS_*` columns are counted back from the application date. When the same person applies twice, `DAYS_BIRTH` changes, but the **differences** `DAYS_BIRTH - DAYS_ID_PUBLISH` and `DAYS_BIRTH - DAYS_REGISTRATION` stay the same. Gender plus these two differences works as a client fingerprint: about 9,650 rows (4%) belong to clients with more than one application, and 1,852 test rows belong to clients that also appear in train.

Features without the target: number of applications, order of this application, days since the previous one and to the next one, change in credit amount.

`CLIENT_OTHER_TARGET` (did this client default on their *other* applications?) uses the target, so it is computed **inside each CV fold** with training-fold labels only (section 6).

In [4]:
def client_key(df):
    """Same person across applications.
    All DAYS_* columns are counted back from the application date, so their
    DIFFERENCES stay constant for the same person in different applications."""
    k1 = df["DAYS_BIRTH"] - df["DAYS_ID_PUBLISH"]
    k2 = (df["DAYS_BIRTH"] - df["DAYS_REGISTRATION"]).round()
    return df["CODE_GENDER"].astype(str) + "_" + k1.astype(str) + "_" + k2.astype(str)


def add_client(df):
    """Repeat-client features (no TARGET involved, safe on train + test)."""
    key = client_key(df)
    df["CLIENT_N_APPS"] = key.map(key.value_counts())
    df["_k"], df["_t"] = key, -df["DAYS_BIRTH"]          # smaller age = earlier application
    df["CLIENT_APP_RANK"] = df.groupby("_k")["_t"].rank(method="first")
    s = df.sort_values(["_k", "_t"])
    g = s.groupby("_k")
    df["DAYS_SINCE_PREV_APP"] = (s["_t"] - g["_t"].shift(1)).reindex(df.index)
    df["DAYS_TO_NEXT_APP"] = (g["_t"].shift(-1) - s["_t"]).reindex(df.index)
    df["PREV_APP_CREDIT_RATIO"] = (s["AMT_CREDIT"] / g["AMT_CREDIT"].shift(1)).reindex(df.index)
    df.drop(columns=["_k", "_t"], inplace=True)

## 4. Model-imputed EXT_SOURCE

`EXT_SOURCE_1/2/3` are the strongest features, but `EXT_SOURCE_1` is missing in 69% of rows. A LightGBM regressor predicts each one from the other columns (no target involved); rows that have the value get an out-of-fold prediction so the feature looks the same for every row.

In [5]:
EXT = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]


def add_ext_pred(df):
    """Predict each EXT_SOURCE from the other columns (no TARGET involved).
    Rows that have the value get an out-of-fold prediction, rows without it get
    a prediction from a model trained on all rows that have it."""
    base = [c for c in df.columns if c not in ("SK_ID_CURR", "TARGET") and c not in EXT]
    Z = df[base].copy()
    for c in Z.select_dtypes(exclude="number").columns:
        Z[c] = Z[c].astype("category")
    reg = dict(n_estimators=400, learning_rate=0.05, num_leaves=31, colsample_bytree=0.5, verbose=-1)
    for col in EXT:
        Zc = pd.concat([Z, df[[c for c in EXT if c != col]]], axis=1)
        has = df[col].notna().values
        idx, pred = np.where(has)[0], np.zeros(len(df))
        for tr, va in KFold(3, shuffle=True, random_state=0).split(idx):
            m = lgb.LGBMRegressor(**reg).fit(Zc.iloc[idx[tr]], df[col].values[idx[tr]])
            pred[idx[va]] = m.predict(Zc.iloc[idx[va]])
        m = lgb.LGBMRegressor(**reg).fit(Zc.iloc[idx], df[col].values[idx])
        pred[~has] = m.predict(Zc.iloc[np.where(~has)[0]])
        df[col + "_PRED"] = pred
        df[col + "_FILLED"] = df[col].fillna(pd.Series(pred, index=df.index))
    df["EXT_FILLED_MEAN"] = df[[c + "_FILLED" for c in EXT]].mean(axis=1)

## 5. Build the feature matrix

In [6]:
def build_xy_fe(groups=(), drop_cols=("FLAG_MOBIL",)):
    """Cleaning + feature groups, applied to train and test together."""
    full = preprocess(pd.concat([train, test], ignore_index=True), drop_cols=drop_cols)
    for g in groups:
        g(full)
    for c in full.select_dtypes(exclude="number").columns:
        full[c] = full[c].astype("category")
    feats = [c for c in full.columns if c not in ("SK_ID_CURR", "TARGET")]
    X = full.loc[: len(train) - 1, feats].reset_index(drop=True)
    X_test = full.loc[len(train):, feats].reset_index(drop=True)
    return X, train["TARGET"], X_test


X, y, X_test = build_xy_fe([add_ratios, add_rates, add_payment_signals, add_client, add_ext_pred])

_keys = client_key(pd.concat([train, test], ignore_index=True)).values
KEY_TR, KEY_TE = _keys[: len(train)], _keys[len(train):]
print("Features:", X.shape[1], "| repeat-client rows in train:", int((X["CLIENT_N_APPS"] > 1).sum()),
      f"| {time.time() - T0:.0f} s")

Features: 62 | repeat-client rows in train: 7180 | 75 s


## 6. Cross-validation

Both models use the same folds. `add_fold_target` adds `CLIENT_OTHER_TARGET`:
* training rows: other applications of the same client inside the training fold (leave-one-out),
* validation rows: applications of the same client inside the training fold only,
* test rows: all train labels.

In [7]:
def other_apps_target(keys_fit, y_fit, keys_apply, exclude_self=False):
    """Mean TARGET of the client's OTHER applications, using only y_fit labels.
    exclude_self=True when keys_apply are the same rows as keys_fit (leave-one-out)."""
    kf, yv = pd.Series(keys_fit), pd.Series(np.asarray(y_fit, dtype=float))
    s, c = yv.groupby(kf.values).sum(), yv.groupby(kf.values).count()
    ka = pd.Series(keys_apply)
    tot, cnt = ka.map(s).fillna(0).values, ka.map(c).fillna(0).values
    if exclude_self:
        tot, cnt = tot - yv.values, cnt - 1
    return np.where(cnt > 0, tot / np.maximum(cnt, 1), np.nan)


FOLDS = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(X, y))


def add_fold_target(Xtr, Xva, tr, va):
    Xtr, Xva = Xtr.copy(), Xva.copy()
    Xtr["CLIENT_OTHER_TARGET"] = other_apps_target(KEY_TR[tr], y.iloc[tr], KEY_TR[tr], exclude_self=True)
    Xva["CLIENT_OTHER_TARGET"] = other_apps_target(KEY_TR[tr], y.iloc[tr], KEY_TR[va])
    return Xtr, Xva


def cv_model(fit_predict, X, X_test, name):
    """fit_predict(Xtr, ytr, Xva, yva, Xte) -> (val_pred, test_pred). Returns OOF, test prediction, fold AUCs."""
    Xte = X_test.copy()
    Xte["CLIENT_OTHER_TARGET"] = other_apps_target(KEY_TR, y, KEY_TE)
    oof, pred_test, fold_auc = np.zeros(len(X)), np.zeros(len(X_test)), []
    for tr, va in FOLDS:
        Xtr, Xva = add_fold_target(X.iloc[tr], X.iloc[va], tr, va)
        p_va, p_te = fit_predict(Xtr, y.iloc[tr], Xva, y.iloc[va], Xte[Xtr.columns])
        oof[va] = p_va
        pred_test += p_te / len(FOLDS)
        fold_auc.append(roc_auc_score(y.iloc[va], p_va))
    auc = roc_auc_score(y, oof)
    print(f"{name:22s} OOF AUC {auc:.4f}  folds {np.round(fold_auc, 4)}  ({time.time() - T0:.0f} s)")
    return dict(auc=auc, oof=oof, pred_test=pred_test, fold_auc=fold_auc)

## 7. LightGBM, 3 seeds

Parameters from the tuning in `main.ipynb`: few leaves (15) and only 20% of the columns per tree, with learning rate 0.02. A learning rate of 0.01 was also tested: CV 0.7692 vs 0.7691, same result at twice the time.

In [8]:
LGB_PARAMS = dict(
    objective="binary", metric="auc", learning_rate=0.02, num_leaves=15, min_child_samples=50,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.2, reg_lambda=1.0,
    n_estimators=5000, n_jobs=-1, verbose=-1,
)


def lgb_fit_predict(seed):
    def f(Xtr, ytr, Xva, yva, Xte):
        m = lgb.LGBMClassifier(**LGB_PARAMS, random_state=seed)
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=[lgb.early_stopping(100, verbose=False)])
        return m.predict_proba(Xva)[:, 1], m.predict_proba(Xte)[:, 1]
    return f


lgb_runs = [cv_model(lgb_fit_predict(s), X, X_test, f"LightGBM seed {s}") for s in SEEDS]
lgb_oof = np.mean([r["oof"] for r in lgb_runs], axis=0)
lgb_test = np.mean([r["pred_test"] for r in lgb_runs], axis=0)
print(f"LightGBM, 3 seeds averaged: OOF AUC {roc_auc_score(y, lgb_oof):.4f}")

LightGBM seed 42       OOF AUC 0.7691  folds [0.7671 0.769  0.7738 0.7647 0.7712]  (222 s)


LightGBM seed 7        OOF AUC 0.7691  folds [0.767  0.7689 0.7738 0.7647 0.7713]  (346 s)


LightGBM seed 2026     OOF AUC 0.7690  folds [0.7675 0.7684 0.7742 0.7638 0.7711]  (474 s)
LightGBM, 3 seeds averaged: OOF AUC 0.7696


## 8. CatBoost, 3 seeds

CatBoost handles the categorical columns with its own target statistics, so its errors are slightly different from LightGBM's. That is what makes the blend useful.

In [9]:
CAT_COLS = X.select_dtypes(["category", "object"]).columns.tolist()
X_cb, X_test_cb = X.copy(), X_test.copy()
for c in CAT_COLS:                                   # CatBoost wants text categories without NaN
    X_cb[c] = X_cb[c].astype("object").fillna("MISSING").astype(str)
    X_test_cb[c] = X_test_cb[c].astype("object").fillna("MISSING").astype(str)


def cb_fit_predict(seed):
    def f(Xtr, ytr, Xva, yva, Xte):
        m = CatBoostClassifier(iterations=5000, learning_rate=0.05, depth=6, eval_metric="AUC",
                               random_seed=seed, od_type="Iter", od_wait=200, verbose=0,
                               cat_features=CAT_COLS)
        m.fit(Xtr, ytr, eval_set=(Xva, yva), use_best_model=True)
        return m.predict_proba(Xva)[:, 1], m.predict_proba(Xte)[:, 1]
    return f


cb_runs = [cv_model(cb_fit_predict(s), X_cb, X_test_cb, f"CatBoost seed {s}") for s in SEEDS]
cb_oof = np.mean([r["oof"] for r in cb_runs], axis=0)
cb_test = np.mean([r["pred_test"] for r in cb_runs], axis=0)
print(f"CatBoost, 3 seeds averaged: OOF AUC {roc_auc_score(y, cb_oof):.4f}")

CatBoost seed 42       OOF AUC 0.7688  folds [0.7682 0.7682 0.7726 0.765  0.7704]  (1384 s)


CatBoost seed 7        OOF AUC 0.7685  folds [0.7672 0.7669 0.7728 0.7655 0.7702]  (2198 s)


CatBoost seed 2026     OOF AUC 0.7684  folds [0.7676 0.7663 0.7728 0.7648 0.7708]  (3042 s)
CatBoost, 3 seeds averaged: OOF AUC 0.7695


## 9. Blend and submission

Any weight between 0.4 and 0.7 gave the same CV, so the blend is a plain 50/50 average: tuning the weight further would only fit noise.

In [10]:
weights = np.round(np.arange(0, 1.01, 0.1), 1)
display(pd.DataFrame({"weight_LightGBM": weights,
                      "CV AUC": [roc_auc_score(y, w * lgb_oof + (1 - w) * cb_oof) for w in weights]}).round(4).T)

W = 0.5
blend_oof = W * lgb_oof + (1 - W) * cb_oof
blend_test = W * lgb_test + (1 - W) * cb_test
blend_folds = [roc_auc_score(y.iloc[va], blend_oof[va]) for _, va in FOLDS]
print(f"Blend 50/50: OOF AUC {roc_auc_score(y, blend_oof):.4f}  folds {np.round(blend_folds, 4)}  "
      f"std {np.std(blend_folds):.4f}")

sub = sample_sub.copy()
assert (sub["SK_ID_CURR"].values == test["SK_ID_CURR"].values).all()
sub["TARGET"] = blend_test
assert sub["TARGET"].between(0, 1).all() and sub["TARGET"].notna().all() and len(sub) == len(sample_sub)
sub.to_csv("output/submission_v8_final.csv", index=False)
print(f"Saved output/submission_v8_final.csv  |  total runtime {(time.time() - T0) / 60:.0f} min")

,0,1,2,3,4,5,6,7,8,9,10
weight_LightGBM,0.0000,0.10,0.2000,0.3000,0.4000,0.5000,0.6000,0.7000,0.8000,0.90,1.0000
CV AUC,0.7695,0.77,0.7703,0.7705,0.7707,0.7707,0.7707,0.7706,0.7703,0.77,0.7696


Blend 50/50: OOF AUC 0.7707  folds [0.7693 0.77   0.7755 0.7665 0.7725]  std 0.0030
Saved output/submission_v8_final.csv  |  total runtime 51 min
